# Migration validation report (Jupyter)

Build dirty data with **Faker**, run `MigrationValidator`, and explore the report as a **pandas DataFrame**.

From the project root:

```bash
uv run jupyter notebook examples/validation_report.ipynb
```

In [1]:
from __future__ import annotations

import random
from pathlib import Path
from typing import Annotated

import pandas as pd
from faker import Faker
from IPython.display import display
from sqlalchemy import Integer, String, create_engine, select
from sqlalchemy.orm import Mapped, Session, mapped_column

from DB.models import Base
from src.engine import MigrationValidator
from src.metadata import FloatMin, Required, Unique
from src.pandas_ import errors_to_dataframe, summarize_errors

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE_PATH = PROJECT_ROOT / "dirty_database_notebook.db"
DATABASE_URL = f"sqlite+pysqlite:///{DATABASE_PATH}"

## Validation model

Rules are declared on columns with `Annotated` markers.

In [2]:
class NotebookUser(Base):
    __tablename__ = "users"
    __table_args__ = {"extend_existing": True}

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    full_name: Mapped[Annotated[str, Required()]] = mapped_column(String(100), nullable=False)
    email: Mapped[Annotated[str, Required(), Unique()]] = mapped_column(String(100), nullable=True)
    age: Mapped[Annotated[int, Required(), FloatMin(0)]] = mapped_column(Integer, nullable=False)
    account_balance: Mapped[Annotated[float, Required(), FloatMin(0.0)]] = mapped_column(
        Integer, nullable=False
    )

## Seed dirty data (Faker)

In [4]:
def generate_dirty_data(row_count: int = 200) -> pd.DataFrame:
    fake = Faker()
    df = pd.DataFrame(
        [
            {
                "full_name": fake.name(),
                "email": fake.email(),
                "age": random.randint(18, 100),
                "account_balance": round(random.uniform(100.0, 10000.0), 2),
            }
            for _ in range(row_count)
        ]
    )
    df.loc[random.sample(range(row_count), 10), "email"] = None
    df.loc[random.sample(range(row_count), 5), "age"] = -15
    return pd.concat([df, df.sample(10)], ignore_index=True)


engine = create_engine(DATABASE_URL, echo=False)
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

dirty_df = generate_dirty_data()
dirty_df.to_sql("users", con=engine, if_exists="append", index=False)

print(f"Loaded {len(dirty_df)} rows into {DATABASE_PATH.name}")
dirty_df.head()

Loaded 210 rows into dirty_database_notebook.db


,full_name,email,age,account_balance
0,Adrian Ritter,daniellesteele@example.net,91,3693.89
1,Kenneth Williamson,olsenstephanie@example.net,56,415.22
2,Andrew Rodriguez,courtneypearson@example.net,60,3069.48
3,Erika Long,barrettleah@example.com,99,2663.96
4,Patrick Smith,vcarney@example.net,20,1296.35


## Run validator and build report DataFrame

In [5]:
with Session(engine) as session:
    row_count = len(session.scalars(select(NotebookUser.id)).all())
    errors = MigrationValidator(session).validate(NotebookUser)

report_df = errors_to_dataframe(errors)
summary_df = summarize_errors(report_df)

print(f"Rows in DB: {row_count}")
print(f"Validation errors: {len(errors)}")

Rows in DB: 210
Validation errors: 38


## Report DataFrame

Each row is one violation — filter, export, or chart from here.

In [6]:
report_df

,table,id,field,error,validator
0,users,38,email,Field 'email' must not be NULL,null
1,users,42,email,Field 'email' must not be NULL,null
2,users,84,email,Field 'email' must not be NULL,null
3,users,108,email,Field 'email' must not be NULL,null
4,users,128,email,Field 'email' must not be NULL,null
5,users,129,email,Field 'email' must not be NULL,null
6,users,132,email,Field 'email' must not be NULL,null
7,users,149,email,Field 'email' must not be NULL,null
8,users,163,email,Field 'email' must not be NULL,null
9,users,198,email,Field 'email' must not be NULL,null


## Summary by rule

In [7]:
summary_df

,table,field,validator,count
0,users,email,duplicate,22
1,users,email,null,10
2,users,age,float_min,6


## Optional: styled view

In [8]:
if not report_df.empty:
    display(
        report_df.sort_values(["validator", "field"]).style.set_properties(
            **{"text-align": "left"}
        )
    )
else:
    print("No validation errors.")

,table,id,field,error,validator
16,users,7,email,Duplicate value 'david55@example.org' for unique field 'email',duplicate
17,users,52,email,Duplicate value 'zjones@example.org' for unique field 'email',duplicate
18,users,59,email,Duplicate value 'scottmcdonald@example.org' for unique field 'email',duplicate
19,users,66,email,Duplicate value 'njones@example.com' for unique field 'email',duplicate
20,users,74,email,Duplicate value 'hpotts@example.org' for unique field 'email',duplicate
21,users,85,email,Duplicate value 'weaverdennis@example.net' for unique field 'email',duplicate
22,users,131,email,Duplicate value 'njones@example.com' for unique field 'email',duplicate
23,users,134,email,Duplicate value 'kenneth25@example.org' for unique field 'email',duplicate
24,users,139,email,Duplicate value 'danielle13@example.org' for unique field 'email',duplicate
25,users,153,email,Duplicate value 'villanuevasteven@example.com' for unique field 'email',duplicate
